In [1]:
import pyspark
from pyspark.sql import SparkSession

spark = SparkSession.builder\
    .appName("test") \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/16 21:56:43 WARN Utils: Your hostname, Adrians-M2-Macbook.local, resolves to a loopback address: 127.0.0.1; using 10.0.0.97 instead (on interface en0)
26/03/16 21:56:43 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/16 21:56:44 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [11]:
df_green = spark.read.parquet('../data/parquet_spark/green/*/*/*.parquet')

26/03/16 22:01:03 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: ../data/parquet_spark/green/*/*/*.parquet.
java.io.FileNotFoundException: File ../data/parquet_spark/green/*/*/*.parquet does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:980)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1301)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:970)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.sinks.FileStreamSink$.hasMetadata(FileStreamSink.scala:58)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:384)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1BatchSource(ResolveDataSource.scala:143)
	at or

In [21]:
columns = [
    'VendorID',
    'lpep_pickup_datetime',
    'lpep_dropoff_datetime',
    'PULocationID',
    'DOLocationID',
    'trip_distance',
    'total_amount'
]

In [22]:
rdd = df_green\
    .select(columns)\
    .rdd

In [13]:
from datetime import datetime
from pyspark.sql import functions as F

In [23]:
start = datetime(2020, 1, 1)

def filter_outliers(row):
    return row.lpep_pickup_datetime >= start


In [24]:
rows = rdd.take(10)
row = rows[0]

In [25]:
row

Row(VendorID=2, lpep_pickup_datetime=datetime.datetime(2020, 1, 11, 4, 5, 54), lpep_dropoff_datetime=datetime.datetime(2020, 1, 11, 4, 13, 49), PULocationID=129, DOLocationID=129, trip_distance=0.81, total_amount=8.51)

In [30]:
def prepare_for_grouping(row):
    hour = row.lpep_pickup_datetime.replace(minute=0, second=0, microsecond=0)
    zone = row.PULocationID
    key = (hour, zone)

    amount = row.total_amount
    count = 1
    value = (amount, count)

    return (key, value)


In [31]:
prepare_for_grouping(row)

((datetime.datetime(2020, 1, 11, 4, 0), 129), (8.51, 1))

In [ ]:
def calculate_revenuue(left_value, right_value):
    left_amount, left_count = left_value
    right_amount, right_count = right_value

    output_amount = left_amount + right_amount
    output_count = left_count + right_count

    return (output_amount, output_count)


In [35]:
from collections import namedtuple

In [36]:
RevenueRow = namedtuple('RevenueRow', ['zone', 'hour', 'revenue', 'count'])

In [37]:
def unwrap(row):
    return RevenueRow(
        zone=row[0][1],
        hour=row[0][0].hour,
        revenue=row[1][0],
        count=row[1][1]
    )
    

In [38]:
from pyspark.sql import types

In [46]:
result_schema = types.StructType([
    types.StructField('hour', types.IntegerType(), True),
    types.StructField('zone', types.IntegerType(), True),
    types.StructField('revenue', types.DoubleType(), True),
    types.StructField('count', types.IntegerType(), True)
])

In [47]:
df_result = rdd\
    .filter(filter_outliers)\
    .map(prepare_for_grouping)\
    .reduceByKey(calculate_revenuue)\
    .map(unwrap)\
    .toDF(result_schema)



In [49]:
df_result.write.parquet('../data/parquet_spark/green_tmp/')

26/03/16 22:22:37 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/03/16 22:22:37 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/03/16 22:22:37 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/03/16 22:22:37 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
26/03/16 22:22:37 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 63.33% for 12 writers
26/03/16 22:22:38 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
26/03/16 22:22:38 WARN MemoryManager: Total allocation exceeds 95.

In [51]:
df_result.show(10)

+----+----+------------------+-----+
|hour|zone|           revenue|count|
+----+----+------------------+-----+
|  39|  13|             171.7|    6|
|  41|   0| 224.6400000000001|   13|
|  52|  18|            215.75|   17|
| 181|  19|            171.49|   15|
| 166|   8|             81.36|    4|
|  40|  10| 48.25999999999999|    4|
|  42|  11|311.60000000000014|   27|
|  17|  15|            135.96|    7|
|  41|  22|185.15000000000006|   17|
|  65|  12|            334.11|   17|
+----+----+------------------+-----+
only showing top 10 rows


Traceback (most recent call last):
  File "/Users/acepeda/Documents/GitHub/zoomcamp-de/06-batch/.venv/lib/python3.11/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 233, in manager
    code = worker(sock, authenticated)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/acepeda/Documents/GitHub/zoomcamp-de/06-batch/.venv/lib/python3.11/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 87, in worker
    outfile.flush()
BrokenPipeError: [Errno 32] Broken pipe


In [52]:
columns = ['VendorID', 'lpep_pickup_datetime', 'PULocationID', 'DOLocationID', 'trip_distance']

duration_rdd = df_green \
    .select(columns) \
    .rdd

In [66]:
duration_rdd.take(5)

[Row(VendorID=2, lpep_pickup_datetime=datetime.datetime(2020, 1, 11, 4, 5, 54), PULocationID=129, DOLocationID=129, trip_distance=0.81),
 Row(VendorID=2, lpep_pickup_datetime=datetime.datetime(2020, 1, 17, 19, 33, 5), PULocationID=75, DOLocationID=42, trip_distance=2.69),
 Row(VendorID=2, lpep_pickup_datetime=datetime.datetime(2020, 1, 30, 12, 41), PULocationID=117, DOLocationID=188, trip_distance=13.11),
 Row(VendorID=2, lpep_pickup_datetime=datetime.datetime(2020, 1, 11, 21, 25, 44), PULocationID=41, DOLocationID=151, trip_distance=2.13),
 Row(VendorID=2, lpep_pickup_datetime=datetime.datetime(2020, 1, 4, 21, 45, 19), PULocationID=129, DOLocationID=260, trip_distance=0.89)]

In [53]:
import pandas as pd

In [54]:
rows = duration_rdd.take(10)

In [58]:
df = pd.DataFrame(rows, columns=columns)

In [60]:
df.head()

,VendorID,lpep_pickup_datetime,PULocationID,DOLocationID,trip_distance
0,2,2020-01-11 04:05:54,129,129,0.81
1,2,2020-01-17 19:33:05,75,42,2.69
2,2,2020-01-30 12:41:00,117,188,13.11
3,2,2020-01-11 21:25:44,41,151,2.13
4,2,2020-01-04 21:45:19,129,260,0.89


In [61]:
def model_predict(df):
    y_pred = df.trip_distance * 5
    return y_pred

In [63]:
def apply_model_in_batch(df):
    df = pd.DataFrame(rows, columns=columns)
    predictions = model_predict(df)

    df['predicted_duration'] = predictions

    for row in df.itertuples():
        yield row


In [67]:
df_predicts = duration_rdd\
    .mapPartitions(apply_model_in_batch)\
    .toDF()\
    .drop('Index')    

In [75]:
duration_rdd.mapPartitionsWithIndex(lambda i, rows: [(i, sum(1 for _ in rows))]).collect() 

[(0, 647086),
 (1, 450535),
 (2, 216387),
 (3, 218240),
 (4, 201259),
 (5, 196764),
 (6, 192505),
 (7, 188063),
 (8, 180018),
 (9, 154497),
 (10, 130844),
 (11, 26733)]

In [76]:
df_predicts.select('predicted_duration').show()

+------------------+
|predicted_duration|
+------------------+
| 4.050000000000001|
|             13.45|
|             65.55|
|10.649999999999999|
|              4.45|
|               4.4|
|             11.25|
|              4.55|
|             13.45|
|             26.75|
| 4.050000000000001|
|             13.45|
|             65.55|
|10.649999999999999|
|              4.45|
|               4.4|
|             11.25|
|              4.55|
|             13.45|
|             26.75|
+------------------+
only showing top 20 rows
